# Reachy 1.2 — Driving the Arm in the FWD Center Lab

A hands-on tutorial for the **FWDCenterLabMCC** scene: the measured MCC tabletop,
the 3×3 taped grid, and the 80/20 rig frame Reachy is bolted into.

Two routines, written to be read and modified:

1. **Place the arm on the table** — out of the rail pocket and onto the board,
   the motion Siva performs by hand (photos in
   `IITG-Reachy-Project/docs/pics/`).
2. **Raise the arm and move the gripper every way it moves** — one joint at a
   time, then in combination.

Each section says *what* the command does and *why* the numbers are what they
are, then runs it. Change the numbers and re-run — that is the point.

---

### Before you start

Launch the simulator on the lab scene so the physics and the RViz view both show
the real geometry:

```bash
REACHY_SIM_SCENE=FWDCenterLabMCC ./scripts/start_sim.sh
```

Watch **RViz at http://localhost:6080** and the **cameras at
http://localhost:8080** while cells run.

> **This is the simulator.** On the physical robot every motion must go through
> `src/reachy_ai/motion/primitives.py` with `REACHY_ENABLE_MOTION=true` and a
> human operator present. The angles below are a *teaching* device — once a
> routine settles, promote it to a named pose in `primitives.py`.

## 0. The rail pocket, and why the arm has to reverse out of it

Reachy's torso is bolted to a horizontal 80/20 frame. The right arm hangs down
through an opening in that frame — call it the **pocket**. Measured in the scene:

| | |
|---|---|
| right shoulder | world **(0.000, −0.190, 1.000)** |
| elbow, arm hanging at rest | world **(0.000, −0.190, 0.720)** |
| board, 1 in thick | surface z = **0.740**, underside z = **0.7146** |
| rig rail tops | z = **0.7146** — the board rests **on top of** them |
| board's robot-side edge | x = **0.160** *(still unmeasured)* |
| the pocket | x ∈ [−0.368, **0.160**], y ∈ [−0.304, −0.076] |

The pocket is **9 in wide** (y) and roughly 20 in long (x). The elbow sits
**20 mm below** the rail plane — it is *inside* the slot, not above it.

There is **no cross rail in front of the board**: that edge is finished with a
wooden trim strip overhanging open air (`docs/pics/80903693586`), so the pocket's
forward bound is the board itself, not aluminium.

That geometry dictates the escape:

- **Sideways is the short axis.** Swinging the arm out laterally — *abduction*,
  `r_shoulder_roll` — has only 114 mm before the outer rail. The elbow rises
  just 20 mm in the first 22° of roll, so it hits the rail before it clears it.
  Below −17.5° of roll the hanging arm is already in collision at rest.
- **Backwards is the long axis.** Swinging the arm back along the rail —
  *extension*, `r_shoulder_pitch` **positive**, with roll held at **0** — travels
  the long dimension. At +40° the elbow is at (−0.180, −0.190, **0.786**): it has
  moved 180 mm back, risen 66 mm, and **y has not changed at all**. It is now
  above the rail plane and out of the pocket.

Extension is capped around **+45°** with the arm straight — beyond that the
forearm reaches the rig's *back* rail. +40° is used here because it leaves the
most margin overall: at +45° the forearm comes within 1.7 mm of that back rail,
at +40° it has 44 mm.

Once the arm is clear, curling the forearm tight lifts the whole lower arm high
above the rails, and only then can the shoulder swing the folded unit forward and
out over the front rail onto the board.

> Checked exhaustively: with `r_shoulder_roll` pinned at 0 the arm can reach
> 28 044 configurations but **none** of them puts the hand over the table — the
> elbow crosses the board's near edge at z = 0.770 and the upper-arm capsule is
> 35 mm in radius, so it clips by 5 mm. The roll is needed on the way *in*, not
> on the way *out*. Every waypoint below was verified at 0.2° resolution against
> the board, all five rig rails, the pedestal and the robot itself.

## 1. Connect and take stock

In [ ]:
import time
import sys
import pathlib

# Make src/ importable whether this runs in the container or from the repo.
for _c in (pathlib.Path("/opt/src"), pathlib.Path.cwd().parent / "src"):
    if _c.is_dir() and str(_c) not in sys.path:
        sys.path.insert(0, str(_c))

from reachy_sdk import ReachySDK                 # Reachy 1.2 SDK — NOT reachy2_sdk
from reachy_sdk.trajectory import goto
from reachy_sdk.trajectory.interpolation import InterpolationMode
from reachy_ai.motion.safety import gate_check
from reachy_ai.scene.awareness import SceneModel

REACHY_HOST = "localhost"
REACHY_PORT = 50051        # fake_reachy_server.py; the physical robot uses 50055

reachy = ReachySDK(host=REACHY_HOST, sdk_port=REACHY_PORT)
arm = reachy.r_arm
print(f"connected to {REACHY_HOST}:{REACHY_PORT}")
print(f"right arm joints: {', '.join(arm.joints.keys())}")
print(f"safety gate_check(): {gate_check()}")

`gate_check()` returns `True` here because `REACHY_SIM_BACKEND` is a simulation
backend. On hardware it returns `False` until `REACHY_ENABLE_MOTION=true` is set
with an operator present. **Never bypass it.**

### The scene, read from the same YAML the simulator loaded

In [ ]:
SCENE_YAML = next(p for p in (
    pathlib.Path("/opt/scenes/FWDCenterLabMCC.yaml"),
    pathlib.Path.cwd().parent / "scenes" / "FWDCenterLabMCC.yaml",
) if p.exists())

scene = SceneModel.from_yaml(str(SCENE_YAML))
table = scene.table

print(f"scene file      : {SCENE_YAML}")
print(f"table surface z : {scene.table_surface_z:.3f} m")
print(f"table extent    : x {table.center[0]-table.size[0]/2:.3f} .. "
      f"{table.center[0]+table.size[0]/2:.3f}   "
      f"y {table.center[1]-table.size[1]/2:.3f} .. {table.center[1]+table.size[1]/2:.3f}")
print(f"rig rails       : {len([o for o in scene.static_obstacles() if 'rig-frame' in o.tags])}")
print()
print("addressable grid cells (robot's view: row 1 = nearest, col 1 = its left):")
for cid in scene.grid_cells():
    x, y, z = scene.cell_center(cid)
    print(f"   {cid}   x={x:+.4f}  y={y:+.4f}  z={z:.4f}")

Two of those nine — **`cell_r3c1` and `cell_r3c2`** — are out of the right arm's
reach (0.709 m and 0.649 m from the shoulder against a 0.609 m maximum). Siva
confirmed that on the physical robot. Plan tasks around the other seven.

## 2. The joint map and its sign conventions

Eight joints, all commanded in **degrees**. The signs are not all intuitive, so
keep this open while you experiment:

| joint | range | what a **positive** value does |
|---|---|---|
| `r_shoulder_pitch` | −150 … +90 | **extension** — swings the arm *backward* along the rail. Negative is **flexion**, forward. |
| `r_shoulder_roll`  | −180 … +10 | negative is **abduction** — swings the arm out sideways, across the rails |
| `r_arm_yaw`        | −90 … +90  | twists the upper arm about its own axis |
| `r_elbow_pitch`    | −125 … 0   | **only negative** — bends the elbow; 0 is a straight arm |
| `r_forearm_yaw`    | −100 … +100| rotates the forearm (pronate / supinate) |
| `r_wrist_pitch`    | −45 … +45  | tilts the hand up / down relative to the forearm |
| `r_wrist_roll`     | −45 … +45  | rolls the hand about the forearm axis |
| `r_gripper`        | −69 … +20  | **inverted: negative OPENS, positive CLOSES** |

Pitch and roll are the two that matter for getting out of the pocket, and they
are *not* interchangeable: pitch moves along the pocket's 19 in axis, roll across
its 9 in axis.

### The gripper sign is backwards from intuition

Measured from `reachy_1_2.xml`, the **right** gripper's pad gap:

| `r_gripper` | pad gap |
|---|---|
| −69° | 7.4 cm (fully open) |
| −45° | 6.5 cm |
| 0°   | 2.6 cm |
| +20° | 0.7 cm (closed) |

The **left** gripper's range is mirrored, so its signs are the opposite way
round. `primitives.open_gripper()` / `close_gripper()` take a `side=` argument
and handle that for you.

In [ ]:
OPEN = -45.0   # gripper open   (~6.5 cm pad gap)
SHUT =  20.0   # gripper closed (~0.7 cm)

R_JOINTS = ["r_shoulder_pitch", "r_shoulder_roll", "r_arm_yaw", "r_elbow_pitch",
            "r_forearm_yaw", "r_wrist_pitch", "r_wrist_roll", "r_gripper"]
ARM7 = R_JOINTS[:7]

from reachy_ai.motion.kinematics import (CartesianPlanner, R_ARM_JOINTS,
                                         UnreachableError, link_capsules,
                                         link_frames)

planner = CartesianPlanner(arm, scene, side="right")


def arm_q():
    """The seven arm angles, in the order the kinematics module expects."""
    return [getattr(arm, j).present_position for j in R_ARM_JOINTS]


def gripper_world_xyz():
    """Gripper pad position in world coordinates, via the SDK's own FK."""
    return planner.fk_world(arm_q())


def elbow_world_xyz():
    """Elbow position in world coordinates.

    Worth having next to the pad, because the two disagree about how safe a
    pose is.  The SDK's FK reports one point on the robot — the wrist, which we
    offset to the pad — and a pose that holds the pad high can still be resting
    the elbow on the table.  See PRESENT in the next cell for what that cost.
    """
    return tuple(float(v) for v in link_frames(arm_q())[1])


def clearance(joints=None, statics=True):
    """Closest approach between ANY arm link and anything it must not touch.

    Negative means the model says they overlap.  ``statics`` includes the table
    and the rig rails as well as the four objects on the board.
    """
    return planner.clearance(arm_q() if joints is None else joints,
                             include_static=statics)


def show_pose(label=""):
    print(label)
    print("   " + "  ".join(f"{j[2:]}={getattr(arm, j).present_position:+6.1f}"
                            for j in R_JOINTS))


reachy.turn_on("r_arm")
time.sleep(0.5)
show_pose("arm powered on, current pose:")
print(f"   gripper pad at world {tuple(round(v, 3) for v in gripper_world_xyz())}")
print(f"   elbow       at world {tuple(round(v, 3) for v in elbow_world_xyz())}")
print(f"   nearest approach     {clearance()}")

### How to actually command a move

Two things about the MuJoCo physics backend, both learned the hard way:

- **Setting `goal_position` once does nothing.** Under `mujoco-remote` the arm
  only tracks while setpoints are being *streamed*. Holding a goal for 30 s
  leaves the arm exactly where it started.
- **Hand-rolled 25 Hz interpolation tracks badly.** Ramping all joints in a
  Python loop leaves 20–60° of steady-state error — the elbow sags out of its
  fold entirely, which wrecks a route that depends on staying folded.

Use the SDK's own `goto()` with minimum-jerk instead. It tracks to **≈2°** on
every joint. Because minimum jerk applies one shared time profile to all joints,
the *path through joint space is still the straight line* between waypoints —
which is exactly what the collision verification below assumes.

`move_to` also **checks that each waypoint was actually reached** before moving
on, and raises `TrackingError` if not. That matters here: the corridor out of the
pocket is a few millimetres wide, so a waypoint reached 10° short is no longer
the pose that was verified, and continuing from it is how the arm ends up jammed
against a rail. A `TrackingError` is the routine refusing to crash the arm — give
that segment a longer duration and re-run.

**Two thresholds, deliberately separate.** `converge_tol` decides how hard
`move_to` tries — how close is close enough to stop re-streaming setpoints —
and defaults to `TRACK_TOL` always. `tol` decides whether falling short is
fatal, and that is the one to loosen when the arm is in open air.

They used to be a single parameter, and merging them was a bug with teeth: the
retry loop exited on the same test that triggered the raise, so passing the loose
`LESSON_TOL` to skip the *abort* also skipped the *retries*. A move got one
0.8 s settle instead of up to four, stopped wherever it had got to, and printed a
line that looked like success. In §4.7 that left the arm 20–42° from its solved
pose, hovering up to 23.6 cm from the cell it had just named — further than the
15.2 cm between cells.

If you loosen a tolerance, be clear which question you are answering: *"don't
abort"* is `tol`, *"don't bother arriving"* is `converge_tol`. They are almost
never the same wish.

In [ ]:
class TrackingError(RuntimeError):
    """The physics arm did not reach a waypoint closely enough to continue."""


# The corridor out of the pocket is only a few millimetres wide, so a waypoint
# that is several degrees short is no longer the pose that was verified.  Rather
# than let that error compound into the next segment (which is how the arm ends
# up jammed against a rail), refuse to continue.
TRACK_TOL = 6.0     # degrees

# Which joints the guard actually polices.  shoulder pitch/roll, arm_yaw, elbow
# and wrist_pitch are what put the elbow, forearm and hand where the clearance
# was measured — those get TRACK_TOL.  r_wrist_roll, r_forearm_yaw and the
# gripper only spin the hand about its own axis; they are weak joints (kp=60,
# 10 Nm) that converge over several waypoints, and a few degrees of error on
# them moves the pad by millimetres inside margins of 20 mm or more.  Holding
# them to 6 deg aborts a route that is in no danger.
CRITICAL = ("r_shoulder_pitch", "r_shoulder_roll", "r_arm_yaw",
            "r_elbow_pitch", "r_wrist_pitch")
LOOSE_TOL = 90.0

# Routine 2 runs at PRESENT, high above the board with the rig far away.  The
# transit guard is there to stop the arm being driven into the rig; it has no
# job here, and aborting would hide the joint behaviour the lesson exists to
# show.  So the sweeps report their tracking error rather than enforcing it.
#
# LESSON_TOL SWITCHES OFF THE ABORT, NOT THE CONVERGENCE.  It used to do both,
# because one `tol` served the retry loop and the raise.  With tol=90 the loop's
# exit test passed on the first pass, so a "lesson" move got ONE 0.8 s settle
# instead of up to four progressively longer ones — and the arm simply stopped
# short.  Measured on the grid sweep in 4.7, that left it 20-42 deg from its
# solved pose and the pad up to 23.6 cm from the cell it had named, while
# printing a line that read like success.  The two thresholds are now separate:
# `converge_tol` decides how hard move_to tries, `tol` decides whether falling
# short is fatal.
LESSON_TOL = 90.0


def joint_error(target, critical_only=True):
    names = [n for n in target if n in CRITICAL] if critical_only else \
            [n for n in target if n != "r_gripper"]
    return max((abs(getattr(arm, n).present_position - target[n]), n) for n in names)


def loose_error(target):
    names = [n for n in target
             if n not in CRITICAL and n != "r_gripper"]
    if not names:
        return (0.0, "")
    return max((abs(getattr(arm, n).present_position - target[n]), n) for n in names)


def clip_target(label, target, margin):
    """Shorten a move until no part of the arm comes within `margin` of anything.

    Returns the target to fly — the original if the whole move is clear, a
    shortened one if it is not, or None if there is nothing safe to fly at all.

    Three things make this a real check rather than the pad-height check it
    replaces, and each corresponds to a way that one was wrong:

    * it uses the WHOLE ARM (three capsules from the MJCF collision geoms), not
      the gripper pad.  At the pose this notebook used to sweep from, the pad
      was 21 cm above the board and the forearm was 5 mm inside red_cube;
    * it checks the PATH, not the endpoints.  `goto` interpolates in joint
      space, so the arm between two clear poses is not itself clear;
    * it starts from where the arm ACTUALLY IS, not from the pose it was last
      told to hold.  Under physics those differ by degrees, and it is the real
      links that have to miss the real cube.

    It refreshes the object positions first, because a guard is only as good as
    its idea of where things are, and objects move — the robot moves them on
    purpose, and it moved them by accident until this notebook was fixed.
    Checking against where an object was PLACED is the same mistake in a
    different costume: the model and the world disagree, and the model wins an
    argument it should lose.

    What it still cannot see is tracking error in the move it is about to make:
    the model knows where the links would be at the commanded pose, not where
    the physics will leave them.  That is what the margin is for, and it is why
    scene_drift() below remains the actual evidence.
    """
    refresh_scene()
    here = arm_q()
    want = dict(zip(R_ARM_JOINTS, here))
    want.update({j: v for j, v in target.items() if j in R_ARM_JOINTS})
    goal = [want[j] for j in R_ARM_JOINTS]
    clipped, frac, c = planner.clip(here, goal, margin=margin, include_static=True)
    if frac > 0.999:
        return target
    if frac < 0.02:
        print(f"  {label:11s} REFUSED — {c}; no part of this move keeps "
              f"{margin * 100:.0f} cm of air")
        return None
    out = dict(target)
    out.update(dict(zip(R_ARM_JOINTS, clipped)))
    changed = ", ".join(f"{j[2:]} {target[j]:+.0f}->{out[j]:+.0f}"
                        for j in target if j in R_ARM_JOINTS
                        and abs(target[j] - out[j]) > 0.5)
    print(f"  {label:11s} CLIPPED to {frac * 100:.0f}% of the way "
          f"({changed}) — {c}")
    return out


def move_to(label, target, secs, report=True, settle=0.8, tol=TRACK_TOL,
            retries=3, converge_tol=None, clear=None):
    """Interpolate to `target` with the SDK's minimum-jerk trajectory generator.

    A short second pass to the same target re-streams the setpoints and pulls out
    the tracking lag; without it the physics arm can finish a fast segment
    10-20 deg short, and simply *holding* a goal will not close the gap — under
    `mujoco-remote` the arm only moves while setpoints are streaming.

    Two independent thresholds, and keeping them apart matters:

    * `converge_tol` — how close is close enough to STOP re-streaming.  Defaults
      to TRACK_TOL whatever `tol` is, so a move always tries just as hard to
      arrive.  Raise it only to deliberately sample a joint mid-flight.
    * `tol` — how far short is far enough to ABORT.  This is the safety guard,
      and it is the one to loosen in open air.

    `clear` is the third, and it guards something the other two cannot see.  A
    move can track perfectly and still be wrong, because tracking says nothing
    about what the arm passes through on the way.  Give it a margin in metres
    and the move is shortened to keep every link that far from every object on
    the board (and from the rig).  Routine 1 leaves it off: that route is
    verified waypoint by waypoint against the rig, and clipping it would abandon
    the corridor it was measured in.

    USE IT ON OUTBOUND MOVES ONLY.  A refusal means "do not move", and that is
    the wrong answer for a move whose purpose is to get OUT of a bad place.
    Observed: with the guard on the return to PRESENT, a run that had already
    disturbed the board refused to retreat — `PRESENT REFUSED, hand clears
    soda_can by 2.7 cm` — and left the arm parked over the wreckage it had just
    made, when the one thing it should have done was leave.  Retreats to a known
    pose go unguarded, deliberately.
    """
    if clear is not None:
        target = clip_target(label, target, clear)
        if target is None:
            return
    cmd = {getattr(arm, n): v for n, v in target.items()}
    ctol = TRACK_TOL if converge_tol is None else converge_tol
    goto(cmd, duration=secs, interpolation_mode=InterpolationMode.MINIMUM_JERK)
    # Progressively longer settles.  The wrist joints are weak (kp=60,
    # forcerange 10) and a single short pass leaves a large offset half-closed.
    for k in range(retries + 1):
        if settle:
            goto(cmd, duration=settle * (1 + k),
                 interpolation_mode=InterpolationMode.MINIMUM_JERK)
        if joint_error(target)[0] <= ctol and loose_error(target)[0] <= LOOSE_TOL:
            break
    err, worst_joint = joint_error(target)
    lerr, ljoint = loose_error(target)
    if lerr > LOOSE_TOL:
        raise TrackingError(
            f"{label}: {ljoint} is {lerr:.1f} deg from its goal (loose tolerance "
            f"{LOOSE_TOL:.0f}).  Even the non-critical joints are not tracking."
        )
    if err > tol:
        raise TrackingError(
            f"{label}: {worst_joint} is {err:.1f} deg from its goal (tolerance "
            f"{tol:.0f}).  The arm is not where the verified route assumes; "
            f"continuing would drive it into the rig.  Re-run the cell, or give "
            f"this segment a longer duration."
        )
    if report:
        x, y, z = gripper_world_xyz()
        print(f"  {label:11s} ({secs:.1f}s)  worst joint error {err:4.1f} deg "
              f"({worst_joint[2:]})   pad -> ({x:+.3f}, {y:+.3f}, {z:.3f})")


LIVE_POSES = "/tmp/reachy_scene_overrides.json"

# Where each object started, captured once.  scene_drift measures against THIS,
# not against the SceneModel — refresh_scene() moves the model to wherever the
# objects actually are, so a model-vs-live comparison would report zero drift by
# construction.  The guard needs to know where things are; the verification
# needs to know where they were.  Different questions, different reference
# points, and collapsing them would quietly disable the only honest check here.
SCENE_ORIGIN = {oid: scene.get(oid).center for oid in scene.manipulable_ids()}


def live_poses(path=LIVE_POSES):
    """The simulator's current object centres, or None if unavailable."""
    import json as _json
    try:
        return _json.load(open(path))
    except Exception:
        return None


def refresh_scene(path=LIVE_POSES):
    """Move the SceneModel's objects to where they actually are right now.

    Under mujoco-remote the container mirrors tracked object poses into `path`
    at 15 Hz.  Without that file the model keeps its as-loaded positions, which
    is the best it can do — and is why a run with no live feed should not be
    trusted to have guarded anything.
    """
    poses = live_poses(path)
    return scene.update_poses(poses) if poses else None


def scene_drift(tag="", path=LIVE_POSES, tol=0.02):
    """Compare LIVE object poses against where they started.

    The counterpart to clip_target(), and the reason both exist.  clip_target
    *predicts* — it is a geometric model of an arm that tracks perfectly, run
    before the move.  This *verifies*, by looking at the objects themselves
    afterwards.  A model can be wrong; the objects cannot.

    Under mujoco-remote the container mirrors tracked object poses into `path`
    at 15 Hz; without that file there is nothing to compare, and we say so
    rather than reporting a clean bill of health.
    """
    live = live_poses(path)
    if live is None:
        print(f"  drift[{tag}]: UNAVAILABLE — cannot confirm the scene is "
              f"undisturbed")
        return None
    moved = []
    for oid in scene.manipulable_ids():
        if oid not in live:
            continue
        want, got = SCENE_ORIGIN[oid], live[oid]
        d = sum((a - b) ** 2 for a, b in zip(want, got)) ** 0.5
        if d > tol:
            moved.append((oid, d))
    if moved:
        print(f"  drift[{tag}]: " + ", ".join(f"{o} moved {d:.3f} m"
                                              for o, d in moved))
    else:
        print(f"  drift[{tag}]: every object still on its cell")
    return moved

## 3. Routine 1 — out of the pocket and onto the board

The motion in four moves, exactly as it is done by hand on the robot:

1. **Back out of the pocket.** `r_shoulder_pitch` → **+40**, roll held at **0**.
   Pure extension along the rail. The elbow travels 180 mm backward and 66 mm up;
   *y never changes*.
2. **Curl the forearm up tight.** Elbow to **−125**, wrist tucked, then extend a
   little further to +70 so the whole folded lower arm rides ~164 mm above the
   rail plane.
3. **Swing the folded unit forward and out over the rail.** Pitch sweeps
   +70 → −17.5 while the roll opens to −37.5, carrying the compact arm over the
   front rail and the table's near edge.
4. **Unfold onto the board.** Elbow opens −120 → −45 and the forearm settles.

| # | pose | pitch / roll | elbow | wrist P/R | grip | elbow world |
|---|---|---|---|---|---|---|
| 0 | `HOME` | 0 / 0 | 0 | 0 / 0 | open | (0.000, −0.190, **0.720**) |
| 1 | `GRIP_SHUT` | 0 / 0 | 0 | 0 / 0 | **shut** | (0.000, −0.190, 0.720) |
| 2 | `BACK` | **+40** / 0 | 0 | 0 / 0 | shut | (−0.180, −0.190, **0.786**) |
| 3 | `CURL` | +40 / 0 | **−125** | +45 / 0 | shut | (−0.180, −0.190, 0.786) |
| 4 | `CURL_HIGH` | **+70** / 0 | −120 | +45 / 0 | shut | (−0.263, −0.190, **0.904**) |
| 5 | `TUCK` | +70 / 0 | −120 | **−45** / 0 | shut | (−0.263, −0.190, 0.904) |
| 6 | `SWING_1` | +37.5 / **−32.5** | −120 | −45 / 0 | shut | (−0.144, −0.340, 0.813) |
| 7 | `SWING_2` | +20 / −35 | −120 | −45 / 0 | shut | (−0.078, −0.351, 0.784) |
| 8 | `SWING_3` | **−17.5** / −37.5 | −120 | −45 / 0 | shut | (+0.067, −0.360, 0.788) |
| 9 | `HOVER` | −40 / −10 | **−60** | −15 / 0 | shut | (+0.177, −0.239, 0.789) |
| 10 | `REST_SHUT` | −40 / −10 | **−45** | −10 / **+30** | shut | (+0.177, −0.239, 0.789) |
| 11 | `REST` | −40 / −10 | −45 | −10 / +30 | **open** | (+0.177, −0.239, 0.789) |

Read the elbow column top to bottom. Rows 0→5 hold **y = −0.190 exactly** — the whole escape happens without a single degree of lateral motion. Only at
row 6, once the elbow is well clear of the rails, does y start to move.

The gripper closes for the transit: an open finger sweeps a wider volume and
catches the front rail. It opens again once the arm is down.

Every segment is contact-free except the last two, which register **−0.97 mm**
against the board. That is the forearm coming to rest — the goal, not a fault.

Measured clearance, worst point in each segment (true surface-to-surface gap):

| segment | gap | closest pair |
|---|---|---|
| HOME → GRIP_SHUT | +79.4 mm | upper arm ↔ inner-right rail |
| GRIP_SHUT → BACK | +44.2 mm | forearm ↔ back rail |
| BACK → CURL | +26.8 mm | thumb pad ↔ board |
| CURL_HIGH → TUCK | +83.9 mm | finger pad ↔ outer-right rail |
| TUCK → SWING_1 | +26.8 mm | forearm ↔ outer-right rail |
| SWING_1 → SWING_2 | +23.4 mm | forearm ↔ board |
| SWING_2 → SWING_3 | +25.9 mm | upper arm ↔ outer-right rail |
| SWING_3 → HOVER | **+4.8 mm** | upper arm ↔ board |

The 4.8 mm is the elbow crossing the board's robot-side edge, and it is the
tightest point on the route. Everything else has centimetres. See §6.

In [ ]:
# ── Verified pose set for FWDCenterLabMCC ────────────────────────────────────
# Checked at 0.2 deg resolution against the board, all five rig rails, the
# pedestal and the robot's own links.  Do not tweak blind — the corridor out of
# the pocket is only a few millimetres wide in places.

def pose(**kw):
    base = dict.fromkeys(ARM7, 0.0)
    base["r_gripper"] = OPEN
    base.update(kw)
    return base


HOME      = pose()
GRIP_SHUT = pose(r_gripper=SHUT)

# 1. back out of the pocket — extension only, roll stays at 0
BACK      = pose(r_gripper=SHUT, r_shoulder_pitch=40.0)

# 2. curl the forearm up tight against the upper arm
CURL      = pose(r_gripper=SHUT, r_shoulder_pitch=40.0,
                 r_elbow_pitch=-125.0, r_wrist_pitch=45.0)
CURL_HIGH = pose(r_gripper=SHUT, r_shoulder_pitch=70.0,
                 r_elbow_pitch=-120.0, r_wrist_pitch=45.0)
TUCK      = pose(r_gripper=SHUT, r_shoulder_pitch=70.0,
                 r_elbow_pitch=-120.0, r_wrist_pitch=-45.0)

# 3. swing the folded unit forward and out over the rail
SWING_1   = pose(r_gripper=SHUT, r_shoulder_pitch=37.5, r_shoulder_roll=-32.5,
                 r_elbow_pitch=-120.0, r_wrist_pitch=-45.0)
SWING_2   = pose(r_gripper=SHUT, r_shoulder_pitch=20.0, r_shoulder_roll=-35.0,
                 r_elbow_pitch=-120.0, r_wrist_pitch=-45.0)
SWING_3   = pose(r_gripper=SHUT, r_shoulder_pitch=-17.5, r_shoulder_roll=-37.5,
                 r_elbow_pitch=-120.0, r_wrist_pitch=-45.0)

# 4. unfold onto the board
HOVER     = pose(r_gripper=SHUT, r_shoulder_pitch=-40.0, r_shoulder_roll=-10.0,
                 r_elbow_pitch=-60.0, r_wrist_pitch=-15.0)
REST_SHUT = pose(r_gripper=SHUT, r_shoulder_pitch=-40.0, r_shoulder_roll=-10.0,
                 r_elbow_pitch=-45.0, r_wrist_pitch=-10.0, r_wrist_roll=30.0)
REST      = dict(REST_SHUT, r_gripper=OPEN)

# Raised pose for routine 2.  Not eyeballed: this is the pose that keeps every
# LINK of the arm clear of every object through every sweep in section 4.
#
# It used to be (-37.5, -2.0, -80.0), described as "collision-free, so joints
# can be swept to their limits".  That was measured on the gripper pad, which
# sat 21 cm above the board and looked entirely safe.  The pad is not the arm:
# at that pose the ELBOW was at z = 0.778, 3.8 cm above a 0.740 tabletop and
# directly over the near-right grid cell where red_cube stands, with the forearm
# already 5 mm INSIDE the cube before any joint moved.
#
# Lifting the hand does not fix that.  The elbow rides a fixed 0.28 m sphere
# about the shoulder; red_cube's nearest surface is 0.320 m from the shoulder
# and the upper arm's own surface reaches 0.315 m, so the near-right cell lies
# inside the elbow's arc no matter how the wrist is held.  The arm has to be
# pointed where the arc does not sweep: up, and out to the right.
#
#     worst clearance, whole arm, over the whole of routine 2:
#         (-37.5,  -2.0, -80.0)   -1.9 cm   already inside red_cube
#         (-70.0, -25.0, -80.0)  +10.9 cm   vs red_cube, via the upper arm
#                                + 8.9 cm   counting the table and rig as well
PRESENT   = pose(r_shoulder_pitch=-70.0, r_shoulder_roll=-25.0,
                 r_elbow_pitch=-80.0)

# (name, pose, seconds, tolerance).  Tolerance is per-waypoint because the
# waypoints are not equally dangerous: settling into GRIP_SHUT happens with
# 79 mm of clearance all round and only has to undo the wrist's gravity drift,
# whereas SWING_3 -> HOVER passes the elbow 4.8 mm from the board's edge.
PLACE_ROUTE = [("GRIP_SHUT", GRIP_SHUT, 3.5, 25.0),
               ("BACK",      BACK,      3.0, TRACK_TOL),
               ("CURL",      CURL,      3.0, TRACK_TOL),
               ("CURL_HIGH", CURL_HIGH, 2.0, TRACK_TOL),
               ("TUCK",      TUCK,      2.0, TRACK_TOL),
               ("SWING_1",   SWING_1,   3.0, TRACK_TOL),
               ("SWING_2",   SWING_2,   2.5, TRACK_TOL),
               ("SWING_3",   SWING_3,   2.5, TRACK_TOL),
               ("HOVER",     HOVER,     2.5, TRACK_TOL),
               ("REST_SHUT", REST_SHUT, 3.0, TRACK_TOL),
               ("REST",      REST,      3.0, TRACK_TOL)]

# The stow route is the placement route run backwards.  Nothing may cut across
# it: a direct move from anywhere over the board to HOME drives the upper arm
# through the board's near edge.
STOW_ROUTE = [("REST_SHUT", REST_SHUT, 2.0, TRACK_TOL),
              ("HOVER",     HOVER,     2.0, TRACK_TOL),
              ("SWING_3",   SWING_3,   2.5, TRACK_TOL),
              ("SWING_2",   SWING_2,   2.5, TRACK_TOL),
              ("SWING_1",   SWING_1,   2.0, TRACK_TOL),
              ("TUCK",      TUCK,      3.0, TRACK_TOL),
              ("CURL_HIGH", CURL_HIGH, 2.0, TRACK_TOL),
              ("CURL",      CURL,      2.0, TRACK_TOL),
              ("BACK",      BACK,      3.0, TRACK_TOL),
              ("GRIP_SHUT", GRIP_SHUT, 3.0, 25.0),
              ("HOME",      HOME,      2.5, 25.0)]


# The joints that decide where the arm sits in the rig.  The wrist angles and
# the gripper do not move the elbow or forearm through the rails, and on a
# freshly reset sim they read wherever gravity left them while the motors were
# off (wrist_roll drifts to ~40 deg, the gripper falls open) — so judging "is
# the arm home?" on those would report a clean reset as a fault.
GROSS_JOINTS = ["r_shoulder_pitch", "r_shoulder_roll", "r_arm_yaw", "r_elbow_pitch"]


def at_pose(target, tol=8.0, joints=None):
    names = joints if joints is not None else [n for n in target if n != "r_gripper"]
    return all(abs(getattr(arm, n).present_position - target[n]) <= tol for n in names)


def pose_distance(target):
    return max(abs(getattr(arm, n).present_position - v)
               for n, v in target.items() if n != "r_gripper")


def ensure_home():
    """Put the arm back in the pocket before starting a placement.

    Makes the placement cell safe to re-run, and safe to run after an aborted
    one.  Jumping straight to HOME from anywhere over the board would cut the
    corner through the rig's front rail, and so would jumping to the *start* of
    the stow route.  Instead: find the waypoint the arm is already closest to,
    ease onto it, and retrace the route from there.
    """
    if at_pose(HOME, joints=GROSS_JOINTS):
        print("arm already at HOME (wrist/gripper will be set by the first move)\n")
        return
    order = [n for n, _, _, _ in STOW_ROUTE]
    by_name = {n: t for n, t, _, _ in STOW_ROUTE}
    nearest = min(order, key=lambda n: pose_distance(by_name[n]))
    print(f"arm is not at HOME (nearest waypoint: {nearest}, "
          f"{pose_distance(by_name[nearest]):.1f} deg away) — retracing from there")
    try:
        # Ease onto the nearest waypoint slowly, with a loose tolerance: this is
        # the one move on an unverified path, so keep it small and gentle.
        move_to(nearest, by_name[nearest], 4.0, report=False, tol=12.0, retries=3)
        for name, target, secs, tol in STOW_ROUTE[order.index(nearest) + 1:]:
            move_to(name, target, secs, report=False, tol=tol)
    except TrackingError as exc:
        raise TrackingError(
            f"cannot recover to HOME: {exc}\n\n"
            "The arm is most likely WEDGED in the rig — an interrupted routine "
            "can leave the forearm threaded under the front rail, where no "
            "commanded pose will pull it back out (the pad ends up below the "
            "tabletop, z < 0.74).  Reset the simulator rather than fighting it:\n"
            "    REACHY_SIM_SCENE=FWDCenterLabMCC ./scripts/start_sim.sh\n"
            "then re-run this notebook from the top."
        ) from None
    print(f"   back at HOME (within 8 deg: {at_pose(HOME, joints=GROSS_JOINTS)})\n")


print(f"{len(PLACE_ROUTE)} placement waypoints, {len(STOW_ROUTE)} stow waypoints")

### Running it — about 35 s. Watch RViz.

In [ ]:
reachy.turn_on("r_arm")
time.sleep(0.5)
ensure_home()

print("out of the pocket and onto the board — watch RViz at localhost:6080\n")
for name, target, secs, tol in PLACE_ROUTE:
    move_to(name, target, secs, tol=tol)
    time.sleep(0.3)

scene_drift("after routine 1")
print("\narm resting on the board.")
show_pose("final pose:")

### What to try

- **Try the sideways version and watch it fail.** Replace `BACK` with
  `pose(r_gripper=SHUT, r_shoulder_roll=-45.0)` — abduction instead of
  extension. Under the MuJoCo backend the arm jams against the outer rail. That
  jamming *is* the feedback; leave the physics backend on rather than switching
  to `kinematic` to make it look clean.
- **Under-curl the elbow.** Set `CURL_HIGH`'s `r_elbow_pitch` to `-100` and the
  forearm drops back toward the rail plane; `SWING_1` then drives it into the
  outer-right rail.
- **Push the extension too far.** `BACK` at `+45` instead of `+40` brings the
  forearm within 1.7 mm of the rig's *back* rail — still legal, but with no
  margin for the physics arm's tracking error. `+50` fouls it outright.
- **Watch the joint errors.** `move_to` prints the worst residual after each
  waypoint. Anything above ~5° means the physics arm did not get where the plan
  assumed, and the clearance numbers no longer apply — which is why `move_to`
  raises `TrackingError` past 6° rather than carrying on.

> **If a routine is interrupted mid-flight**, the arm can end up wedged: the
> forearm threaded under the board's edge, pad below the surface, where no
> commanded pose pulls it back. `ensure_home()` will tell you so. Reset with
> `REACHY_SIM_SCENE=FWDCenterLabMCC ./scripts/start_sim.sh` and start again —
> restarting the native server resets the arm's state.

> **Both routines leave the table alone — and Routine 2 did not used to.**
> Instrumented cell by cell against a freshly reset scene, every object is still
> on its cell after the placement route *and* after the joint sweeps. That was
> not true before: `red_cube` used to start moving in §4.3 and had drifted
> 8.5 cm by §4.5, and §4.7 threw `blue_cylinder` clean off the board. The sweeps
> ran from a pose chosen when the board held nothing, and every check the
> notebook had was watching the gripper pad while the *elbow* did the damage.
> §4 now checks the whole arm along the whole path before each move; see the
> `PRESENT` comment above for the geometry, and `scene_drift()` for the evidence
> that the check is telling the truth.

## 4. Routine 2 — raise the arm, then move the gripper every way it moves

Lift out of the rest position to `PRESENT` and sweep each joint through its
range. Every move in this section is **object-aware**: before it is commanded,
the whole arm — upper arm, forearm and hand, as the three collision capsules the
MJCF actually defines — is swept along the joint-space path the move will fly,
and the move is shortened if any link would come within `SAFE_MARGIN` of
anything on the board.

That guard exists because this section used to sweep the table clean, and the
reason it did is worth reading before the first cell runs.

`PRESENT` used to be `shoulder_pitch −37.5, shoulder_roll −2, elbow −80`, and the
prose here used to say it was "about 21 cm above the board and clear of the rig,
so every angle is collision-free". The 21 cm was true, and it was about the
**gripper pad**. The pad is not the arm. At that same pose the **elbow** sat at
z = 0.778 — 3.8 cm above a 0.740 tabletop, directly over the near-right grid
cell, where `red_cube` stands 6 cm tall. The forearm was already 5 mm *inside*
the cube before any joint moved.

So the cube started drifting in §4.3, a wrist-roll sweep, where the pad never
goes anywhere near it — and by §4.7 the blue cylinder was leaving the table
entirely. Every check the notebook had was watching the pad.

Raising the pad would not have helped, because the elbow is the problem and the
elbow rides a fixed 0.28 m sphere about the shoulder. `red_cube`'s nearest
surface is **0.320 m** from the shoulder; the upper arm's own surface reaches
**0.315 m**. The near-right cell is inside the elbow's arc, and no amount of
lifting the hand changes that. The fix is to point the arm where the arc does
not sweep — up, and out to the robot's right.

| | old `PRESENT` | new `PRESENT` |
|---|---|---|
| shoulder pitch / roll / elbow | −37.5 / −2 / −80 | **−70 / −25 / −80** |
| pad above the board | 0.21 m | 0.50 m |
| elbow above the board | **0.038 m** | 0.17 m |
| worst clearance, whole arm, whole routine | **−1.9 cm** (inside `red_cube`) | **+10.9 cm** |
| …counting the table and rig too | −1.9 cm | +8.9 cm |

In [ ]:
reachy.turn_on("r_arm")
time.sleep(0.3)

# The margin every sweep in this section is held to.
#
# The clearance model is geometry: it knows where each link WOULD be at a
# commanded pose, not where the physics will actually leave it.  Routine 2 from
# PRESENT clears by 8.9 cm at its tightest point, so holding the guard at 5 cm
# leaves ~4 cm for the tracking error the model cannot see, and still leaves
# every intended sweep unclipped.  A clip printed below is therefore news: it
# means the geometry itself, not the margin, ran out of room.
SAFE_MARGIN = 0.05

move_to("PRESENT", PRESENT, 3.0, tol=LESSON_TOL)

px, py, pz = gripper_world_xyz()
ex, ey, ez = elbow_world_xyz()
print(f"\nraised — pad   {pz - scene.table_surface_z:.3f} m above the board")
print(f"         elbow {ez - scene.table_surface_z:.3f} m above the board")
print("         (the elbow is the number that decides whether the table "
      "survives;\n          at the old PRESENT it was 0.038 m)")

print("\nwhole-arm clearance from each object at PRESENT:")
for oid, c in sorted(scene.clearances(link_capsules(arm_q())).items()):
    print(f"   {oid:15s} {c.distance * 100:+6.1f} cm   (nearest link: {c.link})")
print(f"   {'table + rig':15s} "
      f"{clearance().distance * 100:+6.1f} cm   worst of everything")

### 4.1 The gripper itself — aperture

One joint, `r_gripper`, sign inverted: **negative opens**.

In [ ]:
print("gripper aperture — negative opens, positive closes\n")
for label, value in [("fully open", -68.0), ("open (working default)", -45.0),
                     ("half",       -20.0), ("nearly shut",           0.0),
                     ("closed",      20.0)]:
    move_to(f"{value:+.0f}", dict(PRESENT, r_gripper=value), 0.9, report=False,
            tol=LESSON_TOL, clear=SAFE_MARGIN)
    print(f"  r_gripper = {value:+6.1f}   {label}"
          f"   (present {arm.r_gripper.present_position:+6.1f})")
    time.sleep(0.5)
move_to("open", dict(PRESENT, r_gripper=OPEN), 0.9, report=False, tol=LESSON_TOL)

### 4.2 Wrist pitch — tilt the hand up and down

`r_wrist_pitch`, ±45°. This sets the *approach angle* onto the table: positive
tips the hand up, negative down toward the surface. Its narrow range is why
Reachy 1.2 cannot do a straight top-down grasp — with a near-horizontal forearm,
45° is not enough to point the pads at the floor.

In [ ]:
print("r_wrist_pitch — tilt the hand relative to the forearm (±45°)\n")
for v in (0.0, +45.0, 0.0, -45.0, 0.0):
    move_to(f"wp {v:+.0f}", dict(PRESENT, r_wrist_pitch=v), 1.8, tol=LESSON_TOL,
            clear=SAFE_MARGIN)
    time.sleep(0.4)

### 4.3 Wrist roll — rotate the hand about its own axis

`r_wrist_roll`, ±45°. Changes which way the pads face without moving the pad
*position* much. This is how you line the jaws up with an object's long axis.

In [ ]:
print("r_wrist_roll — roll the hand about the forearm axis (±45°)\n")
for v in (0.0, +45.0, 0.0, -45.0, 0.0):
    move_to(f"wr {v:+.0f}", dict(PRESENT, r_wrist_roll=v), 1.8, tol=LESSON_TOL,
            clear=SAFE_MARGIN)
    time.sleep(0.4)
print()
scene_drift("after 4.3")

### 4.4 Forearm yaw — pronate and supinate

`r_forearm_yaw`, ±100°, the widest of the three. It rotates the whole forearm, so
the hand swings through a much larger arc than `r_wrist_roll` gives. Between the
two you can put the jaws at essentially any roll angle you need.

**This joint does not track under physics, and at the raised `PRESENT` it barely
tracks at all.** Measured directly at `PRESENT`, returning to `PRESENT` between
every sample so each one starts from the same place, and taken twice — after one
settle pass and after four — to separate *slow* from *stuck*:

| commanded | after 1 settle | after 4 settles |
|---|---|---|
| +15° | −24.8° | **−40.9°** |
| +45° | −11.3° | −13.0° |
| +90° | +34.9° | +68.5° |
| −45° | −9.9° | **−85.6°** |
| −90° | −59.9° | −85.2° |

Read the middle column against the right one. Extra settling makes `+90°`
better (+34.9 → +68.5) and everything else **worse**: `+15°` ends up 56° the
wrong side of its goal, and `−45°` overshoots to −85.6°. Both negative commands
converge on the same place, ≈ −85°, whatever they were asked for.

The description the data supports is that the joint slides toward ≈ −85°
whenever it is not being driven hard positive, and that more time makes that
worse rather than better. **What causes it is not established here.** The
obvious guess — gravity torque about the forearm's long axis — does not survive
the geometry: that axis is *more* vertical at the new `PRESENT` (29° from
vertical) than at the old one (63°), so the gravity moment about it is smaller,
not larger. `r_forearm_yaw` has `kp=80` against a 15 Nm limit with `kv=5`, and
the actuator model is where to look next.

> **This got worse when `PRESENT` moved.** At the old pose the pattern was
> readable — negative goals were merely slow and arrived given time, positive
> goals stuck. At the raised pose neither half holds. That is a real cost of
> the fix in §4: the new pose is what keeps the arm off the table, and it is
> worse for this joint. Both facts are measured; neither cancels the other.

> **Do not plan a grasp that depends on a precise forearm angle**, positive or
> negative, until the actuator gains are revisited.

> **The sweep below cannot show you this.** `move_to`'s printed "worst joint
> error" comes from `joint_error()`, which only polices the five `CRITICAL`
> joints — and `r_forearm_yaw` is not one of them. It is checked against
> `LOOSE_TOL` (90°), which nothing here trips. So the `fy` lines report
> `elbow_pitch` while the joint the section is about sits 50° from its goal. Read
> `arm.r_forearm_yaw.present_position` directly, as the table above does.

The sweeps in this section **report** their tracking error instead of enforcing
it — they run in open air ~47 cm above the board. What stops them reaching the
objects is not the height but the clearance guard, which checks every link along
every path; see §4's intro.

In [ ]:
print("r_forearm_yaw — pronate / supinate (±100°)\n")
# Sweep to +/-90, not the joint's +/-100 hard stop.  At exactly 100 the
# actuator's ctrlrange and the joint limit are the same number, and the physics
# joint sits tens of degrees short of its goal.
#
# Return to PRESENT between samples.  Without it each reading starts from
# wherever the previous one stranded the joint, and the errors compound into
# nonsense — a sweep straight through 0 -> +90 -> 0 -> -90 -> 0 reported being
# 59.8 deg short of a commanded ZERO, which says nothing about the joint.
# The table above was measured this way, one sample per approach.
#
# Print r_forearm_yaw itself, too: move_to's "worst joint error" covers only the
# five CRITICAL joints and this is not one of them, so the convergence loop is
# blind to it as well — it will stop re-streaming once the CRITICAL joints have
# arrived, however far short this one still is.
for v in (+90.0, -90.0):
    # Unguarded: this is a RETREAT to a pose already known to be clear, and a
    # refused retreat strands the arm exactly where it should not be.
    move_to("PRESENT", PRESENT, 2.0, report=False, tol=LESSON_TOL)
    move_to(f"fy {v:+.0f}", dict(PRESENT, r_forearm_yaw=v), 3.0, report=False,
            tol=LESSON_TOL, clear=SAFE_MARGIN)
    got = arm.r_forearm_yaw.present_position
    verdict = "tracks" if abs(got - v) <= 10 else "DOES NOT CONVERGE"
    print(f"  commanded {v:+6.1f}   reached {got:+6.1f}   "
          f"short by {abs(got - v):5.1f} deg   {verdict}")
    time.sleep(0.4)

move_to("PRESENT", PRESENT, 2.0, report=False, tol=LESSON_TOL)
print(f"\n  back at PRESENT: r_forearm_yaw = "
      f"{arm.r_forearm_yaw.present_position:+.1f} (commanded 0.0)")
scene_drift("after 4.4")

### 4.5 The joints that *translate* the gripper

The three above mostly change the hand's **orientation**. To move the gripper to
a different **place** you drive the big joints. Note how much further the pad
travels per degree here.

These four sweep as **offsets from `PRESENT`**, not to fixed absolute angles.
That is a change worth naming: written absolutely, a sweep stops being "swing
this joint either way from where the arm is" and becomes "drive the arm to this
particular place", which silently breaks the moment the base pose moves. These
same four sweeps, still carrying the absolute numbers that suited the old
`PRESENT`, would fly the arm straight back down onto the board.

In [ ]:
print("the joints that move the gripper's position\n")
# These four sweep RELATIVE to PRESENT, not to fixed absolute angles.
#
# They used to be absolute — arm_yaw to +/-40, shoulder_roll to -20 and +5,
# elbow to -95 and -65, shoulder_pitch to -50 and -25 — which were the right
# numbers only for the PRESENT this notebook started at.  Tie a sweep to an
# absolute angle and it stops being "swing this joint either way from here" and
# becomes "drive the arm to this specific place", which after PRESENT moved
# would have flown the arm back down onto the board on the very first sample.
# Written as offsets they follow the base pose, and the clearance guard has a
# base pose worth guarding.
for joint, deltas in [
    ("r_arm_yaw",        (0.0, -40.0, +40.0, 0.0)),
    ("r_shoulder_roll",  (0.0, -18.0,  +7.0, 0.0)),
    ("r_elbow_pitch",    (0.0, -15.0, +15.0, 0.0)),
    ("r_shoulder_pitch", (0.0, -12.5, +12.5, 0.0)),
]:
    print(f"  {joint}  (about {PRESENT[joint]:+.1f}):")
    for d in deltas:
        move_to(f"{d:+.0f}", dict(PRESENT, **{joint: PRESENT[joint] + d}), 1.4,
                tol=LESSON_TOL, clear=SAFE_MARGIN)
        time.sleep(0.3)
print()
scene_drift("after 4.5")

### 4.6 All together — a wave

Multi-joint targets interpolate simultaneously, which is what makes motion look
deliberate rather than sequential.

In [ ]:
WAVE_A = dict(PRESENT, r_forearm_yaw=-60.0, r_wrist_pitch=25.0, r_wrist_roll=-30.0)
WAVE_B = dict(PRESENT, r_forearm_yaw=+60.0, r_wrist_pitch=-25.0, r_wrist_roll=+30.0)

print("combined motion — watch RViz\n")
for i in range(3):
    # Three weak joints reversing together — 0.9 s per swing left them tens of
    # degrees short and tripped the tracking guard.
    move_to(f"wave {i+1}a", WAVE_A, 1.8, report=False, tol=LESSON_TOL,
            clear=SAFE_MARGIN)
    move_to(f"wave {i+1}b", WAVE_B, 1.8, report=False, tol=LESSON_TOL,
            clear=SAFE_MARGIN)
    print(f"  wave {i + 1}/3")
move_to("PRESENT", PRESENT, 1.8, tol=LESSON_TOL)   # retreat: unguarded
scene_drift("after 4.6")

### 4.7 Pointing at the grid — from joint angles to world coordinates

`CartesianPlanner` wraps the SDK's IK so you can ask for a **world position**
instead of joint angles. It plans in *pad* space (the contact point between the
jaws, ~0.12 m along the wrist's local −Z).

This section is where the object-aware guard stops being reassuring, so read the
cell's comments before the output.

**The IK is not the weak link; the tracking is.** The solver returns a pose that
puts the pad on the target, and the arm then fails to fly it — measured over this
grid, the pad lands up to **21.9 cm** from where it was sent. Joint degrees and
metres at the pad are different questions, and only the second is what "point at
cell r1c2" means.

**That is also what defeats the guard here.** The guard is a *plan-time* check:
it measures the arm at the pose about to be commanded. For the joint sweeps in
4.1–4.6 that is sound — the arm tracks those to ~5°, and three full runs came
back with every object still on its cell. Against a 22 cm Cartesian error, a
5 cm margin means nothing, and this was measured rather than assumed:

> An earlier version of this cell **raised the hover** until each cell passed the
> 5 cm check, then flew them. The guard was clean on every cell it flew. The
> drift check afterwards read `blue_cylinder moved 5.830 m`. Planning clearance
> the arm then fails to fly is *worse* than not flying — it adds moves without
> adding safety.

So the margin here is set from the measured pointing error (20 cm), not from the
geometry. Nothing on this board meets it; the best any cell manages is 13.4 cm.
The cell still runs the lift search, because what it finds is the useful part.
Measured on a fresh scene, the best whole-arm clearance each cell can reach at
any hover between 17 cm and 42 cm:

| cell | best clearance | at hover | limited by |
|---|---|---|---|
| r1c1 | 12.9 cm | 32 cm | `red_cube`, upper arm |
| r1c2 | 12.8 cm | 27 cm | `red_cube`, upper arm |
| r1c3 | 12.8 cm | 27 cm | `red_cube`, upper arm |
| r2c1 | 12.6 cm | 42 cm | `red_cube`, upper arm |
| r2c2 | **9.9 cm** | 42 cm | `red_cube`, upper arm |
| r2c3 | 12.8 cm | 42 cm | `red_cube`, upper arm |
| r3c1 | — | — | unreachable at every height tried |
| r3c2 | 12.6 cm | 42 cm | `red_cube`, upper arm |
| r3c3 | 11.6 cm | 42 cm | `red_cube`, upper arm |

`red_cube` is the limit on **every** cell, and via the **upper arm** every time
— never because the pad goes near it — it stands in the near-right cell, which sits under the
*elbow's* arc for every reach across this board.

**`cell_r3c1` is unreachable**, at every height tried. `cell_r3c2` *does* solve
even though its centre is out of reach on the board — hovering moves the target
up toward the shoulder (z = 1.000), so it is nearer than the cell it is above.
Two cells are unreachable **on the surface**, which is what Siva confirmed on the
physical robot; only one is unreachable from the air.

**The conclusion is a finding, not a bug.** With these four objects standing on
the grid, this arm cannot be trusted to point at a cell: it cannot fly the pose
accurately enough for any achievable clearance to survive the trip. Clear the
board and the section runs. The real fix is closing the loop — move in short
steps and re-check the pose the arm *actually reached* rather than trusting the
one it was asked for. That is not done here.

In [ ]:
# Hover height, derived rather than assumed.  This number is about POINTING;
# it is no longer what keeps the objects on the table.
#
# It was doing both jobs, and doing the second one badly.  The old value added a
# Z_SLOP term of 0.10 m on top of the tallest object, on the reasoning that the
# arm does not fly the height it is given.  That bought nothing: a run with the
# taller hover still threw the blue cylinder 1.06 m off the board, because the
# object was being hit by the FOREARM in transit and no pad height addresses
# that.  What the extra 10 cm did do was wreck the pointing it exists to
# demonstrate — cell_r1c2's miss went from 12.5 cm to 18.8 cm.
#
# Safety is the clearance guard's job now.  The hover goes back to what pointing
# needs: clear the tallest thing standing on the board, plus a little air.
CLEARANCE = 0.06        # air we want under the pad

_tops = [scene.get(o).top_z for o in scene.manipulable_ids()]
_tallest = max(_tops) if _tops else scene.table_surface_z
_stands = _tallest - scene.table_surface_z
HOVER_H = max(0.12, _stands + CLEARANCE)

# THE MARGIN HERE IS FOUR TIMES THE ONE THE SWEEPS USE, AND THAT IS THE POINT.
#
# The guard is a PLAN-TIME check: it measures the arm at the pose that is about
# to be commanded.  That is only worth anything if the arm then flies roughly
# that pose.  For the joint sweeps in 4.1-4.6 it does — measured over three full
# runs, the worst residual was ~5 deg and every drift check came back clean, so
# SAFE_MARGIN = 5 cm covers the gap comfortably.
#
# Cartesian pointing is a different animal.  Over this grid the pad lands up to
# 21.9 cm from where it was sent (cell_r1c2, measured), because 6 deg of joint
# tolerance is centimetres at the end of a half-metre arm.  A 5 cm margin is
# meaningless against a 22 cm error, and this is not theory:
#
#     an earlier version of this cell RAISED the hover until each cell passed
#     the 5 cm check, flew them, and the guard was clean on every one.  The
#     drift check afterwards read "blue_cylinder moved 5.830 m".  Planning
#     clearance the arm then failed to fly is worse than not flying at all,
#     because it adds moves without adding safety.
#
# So POINT_MARGIN is set from the measured pointing error rather than from the
# geometry.  On this board nothing meets it — the best any cell manages is
# 13.4 cm — and the honest consequence is that with these four objects standing
# on the grid, this arm cannot be trusted to point at a cell.  The cell says so
# and flies nothing, which is the version that measured clean.  It still runs
# the search, because "cell_r2c2 would need 38 cm" is the useful part.
#
# The fix is not a bigger number.  It is closing the loop: move in short steps
# and re-check the pose the arm ACTUALLY reached, instead of trusting the one it
# was asked for.  That is real work and it is not done here.
POINT_MARGIN = 0.20
LIFT_STEP = 0.05
LIFT_MAX = 0.25

print(f"tallest object stands {_stands:.3f} m above the board (top z = {_tallest:.3f})")
print(f"base hover {HOVER_H * 100:.0f} cm = {_stands * 100:.0f} object "
      f"+ {CLEARANCE * 100:.0f} clearance; searching up to "
      f"+{LIFT_MAX * 100:.0f} cm for {POINT_MARGIN * 100:.0f} cm of whole-arm "
      f"clearance")
print("(the IK sweeps up to 54 orientations per target — give it a few seconds)\n")

scene_drift("before")
print()

misses, flown, refused = [], [], []
for cid in scene.grid_cells():
    cx, cy, cz = scene.cell_center(cid)
    refresh_scene()
    seed = arm_q()

    chosen, best, reason = None, None, None
    h = HOVER_H
    while h <= HOVER_H + LIFT_MAX + 1e-9:
        target = (cx, cy, cz + h)
        try:
            q = planner.solve(target, seed=seed)
        except UnreachableError as exc:
            reason = f"UNREACHABLE at {h * 100:.0f} cm — {exc}"
            h += LIFT_STEP
            continue
        # `include_static` here means the objects and the RIG RAILS.  It does
        # not mean the tabletop, and this is the section that proves why it must
        # not: pointing at a cell IS reaching across the board, and the upper
        # arm grazes the board's own surface whenever this arm does that.  With
        # the tabletop counted, all nine cells came back blocked and not one of
        # them for a reason involving an object.  See SceneModel.obstacle_ids.
        c = planner.path_clearance(seed, q, include_static=True)
        if best is None or c.distance > best[1].distance:
            best = (h, c)
        if c.distance >= POINT_MARGIN:
            chosen = (h, target, q, c)
            break
        h += LIFT_STEP

    if chosen is None:
        if best is not None:
            reason = (f"best {best[1].distance * 100:.1f} cm at "
                      f"{best[0] * 100:.0f} cm hover ({best[1]})")
        refused.append((cid, reason))
        print(f"  {cid}   NOT FLOWN — {reason}")
        continue

    h, target, q, c = chosen
    move_to(cid, dict(zip(R_ARM_JOINTS, q)), 1.8, report=False, tol=LESSON_TOL)
    px, py, pz = gripper_world_xyz()
    miss = ((px - target[0]) ** 2 + (py - target[1]) ** 2 + (pz - target[2]) ** 2) ** 0.5
    misses.append((miss, cid))
    flown.append((cid, h, c))
    note = "  <-- off by more than a cell" if miss > 0.1524 else ""
    print(f"  {cid}   pad ({px:+.3f}, {py:+.3f}, {pz:.3f})   "
          f"miss {miss * 100:5.1f} cm   hover {h * 100:.0f} cm{note}")
    # Verify per cell, not once at the end.  The old cell checked afterwards and
    # so kept ploughing: it threw the cylinder in this section and the stow then
    # dragged it 2.7 m.  If an object has moved, the model of the board is wrong
    # and every remaining check would be built on it — stop.
    if scene_drift(f"after {cid}"):
        print("   stopping this section: the board is no longer where the "
              "guard thinks it is")
        break
    time.sleep(0.4)

if misses:
    worst, where = max(misses)
    print(f"\nworst miss {worst * 100:.1f} cm at {where}   "
          f"(grid pitch is 15.2 cm — a miss above that is the wrong cell)")
if refused:
    print(f"\n{len(refused)} of {len(scene.grid_cells())} cells not flown:")
    for cid, reason in refused:
        print(f"   {cid}   {reason}")
    print(f"\nThat is the finding, not a failure of the cell.  The heights above "
          f"are what each\ncell WOULD need for {POINT_MARGIN * 100:.0f} cm of "
          f"whole-arm clearance; the arm cannot fly them\naccurately enough for "
          f"that clearance to survive the trip.  Clear the board and\nthe "
          f"section runs; leave it occupied and the honest answer is not to try.")

# The guard is a model, and a model can be wrong.  This is not.  Read the two
# together: a clean guard and a clean drift check mean the geometry and the
# physics agree; a clean guard and a dirty drift check means the model is
# missing something and the model is what needs fixing.
print()
scene_drift("after")

# Retreat — deliberately unguarded.  A refused retreat strands the arm exactly
# where it should not be; see move_to's docstring.
move_to("PRESENT", PRESENT, 2.0, tol=LESSON_TOL)

## 5. Stow — reverse the route back into the pocket

Retrace the placement route backwards. Do not shortcut it: going straight from
`PRESENT` to `HOME` drives the upper arm through the rig's front rail.

In [ ]:
print("stowing — back into the pocket\n")
for name, target, secs, tol in STOW_ROUTE:
    move_to(name, target, secs, tol=tol)
    time.sleep(0.3)

scene_drift("after stow")
reachy.turn_off("r_arm")
print(f"\narm at rest in the rail pocket (at HOME: {at_pose(HOME, joints=GROSS_JOINTS)}), motors off.")

## 6. Practice

1. **Re-time the placement.** Halve every duration in `PLACE_ROUTE` and watch the
   residual joint errors `move_to` prints grow. Above ~5° the verified clearances
   stop applying.
2. **Rest the hand elsewhere on the board.** `REST` puts the pad at
   (0.539, −0.249). Shift `r_shoulder_roll`, check with `gripper_world_xyz()`,
   and keep the whole hand inside y ∈ [−0.349, +0.349].
3. **Touch a cell instead of hovering.** Drop `HOVER_H` to 0.02 and see which
   cells still solve.
4. **Write your own routine** as `(name, pose, duration)` tuples, and verify it
   offline before running — the MuJoCo contact check used to build this notebook
   is in `native_mujoco/`.
5. **Promote what works** into `src/reachy_ai/motion/primitives.py`. That is the
   only place motion may come from on the physical robot.

### The open questions this notebook exposes

Two of the three rig numbers `scenes/FWDCenterLabMCC.yaml` used to carry as
*assumed* have been settled from the photographs and the operator, and both were
wrong in the same direction — they put aluminium where there is none:

- **The rails' height.** The board **rests on top of** the frame
  (`docs/pics/80903696209` shows its laminate edge proud of the rail beneath it).
  The rails had been modelled flush with the board's *surface*, putting 25 mm of
  phantom aluminium in the plane the arm has to cross.
- **The front cross rail.** There isn't one. That edge is finished with wooden
  trim overhanging open air (`docs/pics/80903693586`). A `rig_rail_front` had
  been modelled squarely across the arm's route — the single largest obstruction
  in the scene.

Correcting both transformed the swing: its tightest points went from 1.4 mm and
0.3 mm to 23 mm and 4.8 mm.

**One number is still open, and it is now the binding constraint** — the board's
**robot-side edge**, at x = 0.160 in the scene and never measured. The elbow
crosses it at z = 0.770 carrying a 35 mm collision radius, which is the 4.8 mm in
the clearance table above. At x = 0.190 it would clear by 20 mm instead.

Removing the phantom front rail also removed this edge's lower bound, so it is no
longer even bracketed. A related loose end: with no front member the pocket
measures ~20.8 in fore-aft rather than the 19 in the setup notes record — so
either a front member sits further forward than the photos show, or that edge is
closer to the robot than 0.160. One tape measure from the pedestal axis to the
board's near edge settles both.